# 01 — Train/Test Split in Scikit-Learn

**Goal:** Learn why we divide a dataset into training and testing sets, how to use `train_test_split()`, and how to evaluate a model on unseen data.

**Prerequisites:** Basic NumPy, `fit()`, `predict()`, and `score()`.

## 1. Why split the data?

- **Training data:** Examples the model learns from using `fit()`.
- **Testing data:** Examples held back until evaluation to check how the model performs on unseen data.

Evaluating only on training data can give an overly optimistic result. A common starting split is **80% training / 20% testing**, but the appropriate ratio depends on dataset size and the task.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

## 2. Create a simple dataset

Suppose `X` is hours studied and `y` is an exam score. Scikit-Learn expects feature data `X` to be two-dimensional: `(number_of_samples, number_of_features)`.

In [ ]:
X = np.array([[1], [2], [3], [4], [5], [6], [7], [8], [9], [10]])
y = np.array([12, 19, 31, 39, 52, 58, 72, 79, 91, 98])

print('X shape:', X.shape)
print('y shape:', y.shape)

## 3. Split the dataset

`test_size=0.2` reserves 20% of examples for testing. `random_state=42` makes the random split reproducible; **42 is arbitrary**. `train_test_split` shuffles samples by default, keeping each feature row matched to its target.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('X_train:', X_train.ravel())
print('X_test: ', X_test.ravel())
print('y_train:', y_train)
print('y_test: ', y_test)
print('Training shape:', X_train.shape)
print('Testing shape: ', X_test.shape)

# random_state=None (default):
# Data is randomly split each time, so different rows may be selected.
# The number of rows stays the same (80 training, 20 testing).

# random_state=5:
# The same rows are selected for training and testing every time
# you run the code with the same dataset and settings.
# Any integer (5, 7, 42, etc.) can be used as a random seed.
# Different seeds can produce different splits.
# It does NOT control how many rows the model trains on.
# It does NOT control training iterations.
# It only controls the random splitting of data.

### What do these four variables mean?

| Variable | Meaning |
|---|---|
| `X_train` | Input features used for training |
| `X_test` | Input features held out for testing |
| `y_train` | Correct target values for training |
| `y_test` | Correct target values for testing |

**Important:** Do not train the model on `X_test` or `y_test`.

## 4. Train on the training set only

`fit(X_train, y_train)` learns the relationship from training examples.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print('Coefficient:', model.coef_)
print('Intercept:', model.intercept_)

## 5. Predict on unseen test data

Compare predictions with the held-out correct answers.

In [ ]:
y_pred = model.predict(X_test)

for hours, actual, predicted in zip(X_test.ravel(), y_test, y_pred):
    print(f'Hours: {hours}, Actual: {actual}, Predicted: {predicted:.2f}')

# ravel() = make it 1D, avoiding a copy when possible. flatten() = make it 1D, always creating a new copy.    
# ravel() and flatten() are NumPy methods used to convert a multidimensional array into a 1D array.

# zip() combines corresponding elements from multiple iterables.
# It returns an iterator of tuples.
# By default, it stops at the shortest iterable.

## 6. Evaluate the model

For `LinearRegression`, `.score()` returns **R²**, not accuracy. R² = 1 indicates perfect predictions; it can also be negative. With only two test examples, this score is unstable, so do not treat this tiny demonstration as a reliable performance estimate.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('Training R²:', model.score(X_train, y_train))
print('Testing R²: ', model.score(X_test, y_test))
print('Test MAE:   ', mean_absolute_error(y_test, y_pred))
print('Test MSE:   ', mean_squared_error(y_test, y_pred))

## 7. Visualize the training and testing data

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(X_train.ravel(), y_train, label='Training data')
plt.scatter(X_test.ravel(), y_test, label='Testing data')

x_line = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
plt.plot(x_line.ravel(), model.predict(x_line), label='Regression line')
plt.xlabel('Hours studied')
plt.ylabel('Exam score')
plt.title('Train/Test Split')
plt.legend()
plt.show()

# linspace(start, stop, num):
# Creates 'num' evenly spaced values from start to stop.

# reshape(rows, columns):
# Changes the shape of an array without changing its elements.

# reshape(-1, 1):
# Converts a 1D array into a 2D column array.

# reshape(1, -1):
# Converts a 1D array into a 2D row array.



- Try `shuffle=False`. What changes?


 shuffle=True (default):
 Randomly mixes the rows before splitting.



 shuffle=False:
 Keeps the original row order.
 The last 20% becomes testing data if test_size=0.2.



 random_state=42:
 Keeps the random split the same every time.
 Use it when shuffle=True.




## R² (R-Squared) Formula

$$
\boxed{R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}{\sum_{i=1}^{n}(y_i-\bar{y})^2}}
$$

**Where:**

- $y_i$ = Actual value
- $\hat{y}_i$ = Predicted value
- $\bar{y}$ = Mean of actual values
- $n$ = Total number of data points

R² measures how much of the variation in actual target values our regression model explains, compared with predicting the average value. Its maximum value is 1, it can be 0, and it can also be negative.

A high R² does not necessarily mean every individual prediction is accurate. For that, you can also examine MAE and RMSE.